# MetaPub

## Documentation links
- https://metapub.org
- https://metapub.readthedocs.io/en/latest/examples.html
- https://metapub.readthedocs.io/en/latest/pubmedarticle_properties.html

## Notes
- Using an API key requires env variable to be set
  - `export NCBI_API_KEY="Your_key_here"`
- Currently using local LLM at url in code below (Use Ollama, LMStudio, etc)

In [94]:
from metapub import PubMedFetcher
import requests
import json

In [95]:
# first patient interaction
# patient_statement_01 = 'I have a headache'
# patient_statement_01 = 'I have a stomach ache that has been there for two weeks and it hurts more after I eat'
# patient_statement_01 = 'I have been sneezing and I have a runny nose for two weeks'
patient_statement_01 = 'I have stomach pain and it hurts more after I eat'
# patient_statement_01 = 'I have a stomach ache and it hurts more after I eat'

In [96]:
# use LLM to pull out potential symptom information only
def query_llm_symptom_check(prompt):
    """Queries the LLM API."""
    url = "http://127.0.0.1:1234/v1/chat/completions"
    headers = {
        "Content-Type": "application/json"
    }

    # look for symptom information
    prompt = f'List keywords for medical symptoms in this statement with bullet points and no additional information: {prompt}'

    data = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

# retreive relevant topics to send to PubMed
topic = query_llm_symptom_check(patient_statement_01)

In [97]:
# Initialize the fetcher
fetch = PubMedFetcher()

# Let user know what's going on
print(f'Finding articles on:\n{topic}')

# Search for articles
pmids = fetch.pmids_for_query(topic, retmax=10)

# Get article details
for pmid in pmids:
    article = fetch.article_by_pmid(pmid)
    print(f"{article.title}")
    print(f"Authors: {', '.join(article.authors)}")
    print(f"Journal: {article.journal}")
    print("---")

Finding articles on:
* Stomach pain
* Pain
* Eating
* Postprandial (after eating)

Chronic Mesenteric Ischemia
Authors: Patel R, Waheed A, Kimyaghalam A, Costanza M
Journal: StatPearls
---
Postprandial Referred Shoulder Pain: A Case Report.
Authors: Ott K, Iwanaga J, Dumont AS, Loukas M, Tubbs RS
Journal: Cureus
---
Association Between Food Intake and Gastrointestinal Symptoms in Patients With Obesity.
Authors: Ghusn W, Cifuentes L, Campos A, Sacoto D, De La Rosa A, Feris F, Calderon G, Gonzalez-Izundegui D, Stutzman J, Hurtado MD, Camilleri M, Acosta A
Journal: Gastro Hep Adv
---
Chronic gastrointestinal ischaemia: shifting paradigms.
Authors: Mensink PB, Moons LM, Kuipers EJ
Journal: Gut
---
Possible relation between partial small bowel obstruction and severe postprandial reactive hypoglycemia after Roux-en-Y gastric bypass.
Authors: Laurenius A, Hedberg S, Olbers T
Journal: Surg Obes Relat Dis
---
Patients with eating disorders showed no signs of coeliac disease before and after nut